<a href="https://colab.research.google.com/github/23151651/ISYS2001-23151651/blob/main/ISYS2001A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Final product

In [ ]:
import pandas as pd
from google.colab import files
from datetime import datetime

def generate_timestamp():
  """Generate the current date and time."""
  return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def upload_csv():
  """Allow user to upload CSV."""
  print("\nPlease select a CSV file. ")
  uploaded = files.upload()
  filename = next(iter(uploaded))
  encodings = ["utf-8", "cp1252", "latin1"]
  df = None
  for encoding in encodings:
    try:
      df = pd.read_csv(filename, encoding=encoding)
      print(f"\nFile read" f"using {encoding} encoding.")
      break
    except UnicodeDecodeError:
      continue
  if df is None:
      print("\nError: CSV could not be read. ")
      return None
  required_columns = ["Date", "Category", "Description", "Type", "Amount"]
  if not all(column in df.columns for column in required_columns):
      print("\nError: CSV is missing required columns. ")
      return None
  df = df[required_columns].copy()

  df["Amount"] = (df["Amount"].astype(str).str.replace("$","", regex=False).str.replace(",","", regex=False).str.strip())
  df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
  df["Type"] = (df["Type"].astype(str).str.strip().str.title())
  df["Category"] = (df["Category"].astype(str).str.strip())
  df = df.dropna(subset=["Date", "Amount"])
  df = df[df["Type"].isin(["Income", "Expense"])]
  df = df[df["Amount"] > 0]
  return df

def manual_input():
    """Allow the user to manually enter transactions."""
    transactions = []
    print("\nMANUAL FINANCE INPUT")
    print("-" * 40)
    while True:
        transaction_type = input(
            "\nIs this Income or Expense? "
            "(or type 'done' or 'finish' to finish): ").strip().lower()
        if transaction_type in ["done", "finish"]:
            print("\nFinished entering transactions." )
            break
        transaction_type = transaction_type.title()
        while transaction_type not in [
            "Income",
            "Expense"]:
            print(
                "Please enter either "
                "'Income' or 'Expense'.")
            transaction_type = input(
                "Is this Income or Expense? "
            ).strip().lower()
            if transaction_type in [
                "done",
                "finish"]:
                print("\nFinished entering transactions.")
                return pd.DataFrame(
                    transactions)
            transaction_type = (
                transaction_type.title())
        while True:
            try:
                amount = float(
                    input("Enter the amount: $")
                )
                if amount <= 0:
                    print(
                        "Please enter a "
                        "positive amount.")
                    continue
                break
            except ValueError:
                print("Please enter a valid number.")
        category = input("Enter the category: ").strip()
        description = input("Enter a description: ").strip()
        transactions.append({
            "Date": datetime.now().strftime(
                "%Y-%m-%d"),
            "Category": category,
            "Description": description,
            "Type": transaction_type,
            "Amount": amount})
        print("\nTransaction added successfully!")
    df = pd.DataFrame(transactions)
    return df

def display_finance_analysis(df):
  """Calculate and Display Financial Information."""
  if df.empty:
    print("\nNo transactions were entered.")
    return
  income = df.loc[df["Type"] == "Income", "Amount"].sum()
  expenses = df.loc[df["Type"] == "Expense", "Amount"].sum()
  balance = income - expenses
  print("\nTransaction History")
  print("="*50)
  print(df.to_string(index=False))
  print("\nFinancial Summary")
  print("="*50)
  print(f"Total Income: ${income:,.2f}")
  print(f"Total Expenses: ${expenses:,.2f}")
  print(f"Remaining Balance: ${balance:,.2f}")
  print("\nSpending by Category")
  print("="*50)
  spending = df[df["Type"] == "Expense"]
  category_totals = spending.groupby("Category")["Amount"].sum().sort_values(ascending=False)
  if category_totals.empty:
    print("No expenses recorded.")
  else:
    for category, amount in (category_totals.items()):
      if expenses > 0:
        percentage = (amount / expenses) * 100
      else:
        percentage = 0
      print(f"{category}: "f"${amount:,.2f} " f"({percentage:.1f}%)")
  print("\nFinancial Status")
  print("="*50)
  if balance > 0:
    print("You're on your way to saving up!")
  elif balance == 0:
    print("You're spending as much as you earn.")
  else:
    print("You're losing money! Maybe cut down on some things.")

def main():
  """Main Program"""
  print("="*50)
  print("Personal Finance Tracker")
  print("="*50)
  timestamp = generate_timestamp()
  print(f"Beginning Session at: {timestamp}")
  print()
  name = input("What's your name?")
  print(f"\nWelcome {name} to your new Personak Finance Tracker! ")
  print(f"\nWe are glad to have you.")
  while True:
    csv_choice = input("\nWould you like to import a CSV file? (yes/no)").strip().lower()
    if csv_choice in ["yes", "no"]:
      break
    print("Please enter yes or no")
  if csv_choice == "yes":
    df = upload_csv()
    if df is None:
      print("\nThe CSV could not be imported")
      return
  else:
    df = manual_input()

  display_finance_analysis(df)
  save_choice = input("\nWould you like to save your transactions into a CSV file? ").strip().lower()
  if save_choice == "yes":
    output_file = "finances.csv"
    df.to_csv(output_file, index=False)
    print(f"\nYour file has been downloaded as {output_file}.")
    files.download
  print("\nThank you for using the Personal Finance Tracker! :D ")

main()